# Landing: canonical sources

**Audience:** data engineers validating medallion architecture and AIDP lineage.

**Prerequisites:** the canonical lab assets, shared compute and five job parameters.

**Learning goals:** trace governed transformations, verify isolation, and inspect deterministic results.


In [ ]:
import re
from functools import reduce
from pyspark.sql import Window, functions as F

# oidlUtils is injected by AIDP Workbench; no import is required.
def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != "telco_lineage":
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

layer_prefixes = {"landing": "01_landing", "bronze": "02_bronze", "silver": "03_silver", "gold": "04_gold"}

def table(layer, logical_name):
    return f"aidp_lab.oci_{layer}.{participant_key}_{lab_id}_{logical_name}"

def location(layer, logical_name):
    return f"oci://{bucket_name}@{objectstorage_namespace}/{layer_prefixes[layer]}/users/{participant_key}/{lab_id}/{logical_name}/"

def write_delta(frame, layer, logical_name, _ddl):
    target = table(layer, logical_name)
    target_location = location(layer, logical_name)
    (frame.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").option("path", target_location)
        .saveAsTable(target))
    actual = spark.table(target).count()
    assert actual == frame.count(), f"Delta count mismatch for {logical_name}"
    print(f"Delta {layer}.{logical_name}: {actual} rows")


## Transformation

Run this cell once. It is idempotent and checks its row-level contract.


In [ ]:
from pathlib import Path
from pyspark.sql.types import StringType, StructField, StructType

dataset_columns = {'crm_customers': ['source_row_id', 'customer_id', 'first_name', 'last_name', 'document_number', 'segment', 'status', 'updated_at'], 'crm_addresses': ['source_row_id', 'address_id', 'customer_id', 'address_type', 'address_line', 'city', 'province', 'region', 'is_primary', 'updated_at'], 'product_catalog': ['source_row_id', 'product_id', 'service_type', 'product_name', 'product_family', 'monthly_fee', 'status', 'updated_at'], 'prepaid_lines': ['source_row_id', 'line_id', 'customer_id', 'product_id', 'msisdn', 'activation_date', 'status', 'updated_at'], 'prepaid_recharges': ['source_row_id', 'recharge_id', 'line_id', 'recharge_date', 'channel', 'amount', 'updated_at'], 'postpaid_accounts': ['source_row_id', 'account_id', 'customer_id', 'billing_cycle', 'status', 'updated_at'], 'postpaid_lines': ['source_row_id', 'line_id', 'account_id', 'product_id', 'msisdn', 'activation_date', 'status', 'updated_at'], 'postpaid_invoices': ['source_row_id', 'invoice_id', 'account_id', 'invoice_month', 'amount', 'status', 'updated_at'], 'home_services': ['source_row_id', 'service_id', 'customer_id', 'product_id', 'service_number', 'activation_date', 'status', 'updated_at'], 'home_installations': ['source_row_id', 'installation_id', 'service_id', 'address_id', 'technology', 'installed_at', 'updated_at']}
expected_counts = {'crm_customers': 500, 'crm_addresses': 620, 'product_catalog': 15, 'prepaid_lines': 650, 'prepaid_recharges': 3000, 'postpaid_accounts': 280, 'postpaid_lines': 400, 'postpaid_invoices': 1500, 'home_services': 220, 'home_installations': 220}
source_root = Path(workspace_root) / "source"
assert {path.name for path in source_root.glob("*.csv")} == {f"{name}.csv" for name in dataset_columns}

for dataset, columns in dataset_columns.items():
    schema = StructType([StructField(name, StringType(), True) for name in columns])
    source = (spark.read.option("header", True).schema(schema)
        .csv(str(source_root / f"{dataset}.csv"))
        .withColumn("participant_key", F.lit(participant_key))
        .select("participant_key", *columns))
    assert source.count() == expected_counts[dataset]
    target_location = location("landing", dataset)
    source.write.mode("overwrite").option("header", True).csv(target_location)
    ddl = ", ".join(["participant_key STRING", *[f"`{name}` STRING" for name in columns]])
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {table('landing', dataset)} ({ddl}) USING CSV OPTIONS (header 'true') LOCATION '{target_location}'")
    assert spark.table(table("landing", dataset)).count() == expected_counts[dataset]
    print(f"CSV landing.{dataset}: {expected_counts[dataset]} rows")


## Exercise and common pitfall

**Exercise:** follow one customer or service identifier into the next task and explain every derived column.

**Answer scaffold:** identify the source table, join key, transformation and target column.

**Pitfall:** never replace the job parameters with participant-specific literals; doing so breaks canonical hashes and isolation.

**Extension:** inspect the resulting entity and column lineage in Master Catalog.
